# County HSDS Download And BA Weather Aggregation

This notebook shows the next step after `county_weather_point_selection.ipynb`: downloading weather variables from those selected HSDS `gid` values, staging them as reusable county-weather H5 files, and population-weighting counties into one BA-level weather CSV. This BA-level weather CSV is then used for load forecasting with TELL via `data_flow/tell_load_forecast_data_flow.ipynb`

Once county-weather H5 files exist for a weather source and year, any BA can be built by selecting that BA's counties and applying normalized county-population weights. This public notebook keeps the example computationally small by staging all counties but only the first `HOUR_LIMIT` hourly positions.

A reference-only appendix documents how WTK `specifichumidity_2m` is derived. The default `MISO` example uses BC-HRRR and NSRDB, so that derivation is not called in the main workflow.

## Imports And Package Setup

Load the Python libraries used below and define the package-relative roots used by the workflow.


In [1]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
from rex import Resource

DATA_ROOT = Path("../data")
OUTPUT_DIR = Path("../notebook_outputs/data_flow/county_hsds_download_and_ba_weather_aggregation")

if not DATA_ROOT.is_dir():
    raise FileNotFoundError(
        "Expected to find the packaged data folder at ../data. "
        "Open this notebook from inside paper_data/data_flow."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: Load Selected County HSDS Points

`county_centroid_regrid.csv` (previously created from `county_weather_point_selection.ipynb`) tells this notebook which HSDS grid id (`gid`) to read for each county and weather source. Each row gives one county, the packaged `weather_source` value identifying the weather source, and the selected `gid` for that weather source.

In [2]:
WEATHER_SOURCES = ["bchrrr", "nsrdb"]  # Sources combined to create the final BA weather schema.

PROVENANCE_DIR = DATA_ROOT / "source_inputs" / "county_weather_provenance"
COUNTY_REGRID_CSV = PROVENANCE_DIR / "county_centroid_regrid.csv"

# Read county_fips as text so leading zeros can be restored/preserved.
regrid_matches = pd.read_csv(COUNTY_REGRID_CSV, dtype={"county_fips": str})
regrid_matches["county_fips"] = regrid_matches["county_fips"].str.zfill(5)
selected_source_matches = regrid_matches[regrid_matches["weather_source"].isin(WEATHER_SOURCES)].copy() # Keep only the selected weather sources in WEATHER_SOURCES.
display(selected_source_matches.head())


,county_fips,county_name,state_name,weather_source,county_pop_lat,county_pop_lon,selected_gid,grid_lat,grid_lon,distance_km
1,01001,Autauga County,Alabama,bchrrr,32.500197,-86.487818,1827777,32.498220,-86.487335,0.224451
2,01001,Autauga County,Alabama,nsrdb,32.500197,-86.487818,926773,32.490000,-86.500000,1.609640
4,01003,Baldwin County,Alabama,bchrrr,30.537375,-87.761515,1772629,30.543240,-87.768130,0.909203
5,01003,Baldwin County,Alabama,nsrdb,30.537375,-87.761515,900257,30.530000,-87.780000,1.951121
7,01005,Barbour County,Alabama,bchrrr,31.844091,-85.301177,1891804,31.841358,-85.310640,0.944123


## Step 2: Stage A 1-Hour National County H5 Example

Before running this step, start the local HSDS service at `http://localhost:5101` with access to the `nrel-pds-hsds` bucket.


In [ ]:
HSDS_ENDPOINT = "http://localhost:5101"
HSDS_API_KEY = None
HSDS_BUCKET = "nrel-pds-hsds"

YEAR = 2023  # In 2023, BC-HRRR supplies most load-weather variables and NSRDB supplies ghi.
HOUR_LIMIT = 1  # Stage only the first top-of-hour timestamp so the worked example stays small.

COUNTY_H5_DIR = OUTPUT_DIR / "county_weather_h5"
COUNTY_H5_DIR.mkdir(parents=True, exist_ok=True)
GID_CHUNK_SIZE = 800

SOURCE_RESOURCE_PATHS = {
    "wtk": f"/nrel/wtk/conus/wtk_conus_{YEAR}.h5",
    "bchrrr": f"/nrel/wtk/bchrrr/v1.0.0/bchrrr_conus_{YEAR}.h5",
    "nsrdb": f"/nrel/nsrdb/GOES/aggregated/v4.0.0/nsrdb_{YEAR}.h5",
}

SOURCE_VARIABLES = {
    "wtk": ["temperature_2m", "windspeed_10m", "relativehumidity_2m", "pressure_0m"],
    "bchrrr": ["temperature_2m", "windspeed_10m", "relativehumidity_2m", "pressure_0m", "specifichumidity_2m"],
    "nsrdb": ["ghi"],
}

NATIVE_WEATHER_COLUMNS = ["temperature_2m", "specifichumidity_2m", "windspeed_10m", "ghi", "relativehumidity_2m", "pressure_0m"]

county_h5_by_source = {}
county_h5_staging_rows = []

for weather_source in WEATHER_SOURCES:
    source_points = selected_source_matches[selected_source_matches["weather_source"].eq(weather_source)].reset_index(drop=True)
    resource_path = SOURCE_RESOURCE_PATHS[weather_source]
    variables = SOURCE_VARIABLES[weather_source]
    output_h5 = COUNTY_H5_DIR / f"{weather_source}_{YEAR}_all_county_weather.h5"
    county_h5_by_source[weather_source] = output_h5
    gids = source_points["selected_gid"].to_numpy(dtype=int)

    # 1. Open this weather source's HSDS resource.
    with Resource(resource_path, hsds=True, hsds_kwargs={"endpoint": HSDS_ENDPOINT, "api_key": HSDS_API_KEY, "bucket": HSDS_BUCKET}) as resource:
        time_index = pd.DatetimeIndex(pd.to_datetime(resource.time_index, utc=True))

        # 2. Keep top-of-hour timestamps for every source. NSRDB is sub-hourly; BC-HRRR is already hourly.
        selected_times = time_index[time_index.minute == 0]
        selected_times = selected_times[:HOUR_LIMIT]
        time_positions = time_index.get_indexer(selected_times)

        # 3. Build this weather source's county-weather H5 staging file.
        with h5py.File(output_h5, "w") as h5_file:
            h5_file.attrs["kind"] = "county_load_weather"
            h5_file.attrs["source"] = weather_source
            h5_file.attrs["hsds_resource_path"] = resource_path
            h5_file.create_dataset("time_utc", data=selected_times.strftime("%Y-%m-%dT%H:%M:%SZ").tolist(), dtype=h5py.string_dtype("utf-8"))
            h5_file.create_dataset("county_fips", data=source_points["county_fips"].tolist(), dtype=h5py.string_dtype("utf-8"))
            h5_file.create_dataset("gids", data=gids, dtype="i8")
            h5_file.create_dataset("latitude", data=source_points["county_pop_lat"].to_numpy(dtype=float), dtype="f8")
            h5_file.create_dataset("longitude", data=source_points["county_pop_lon"].to_numpy(dtype=float), dtype="f8")

            # 4. Save each weather variable as a two-dimensional array: time x county.
            for variable in variables:
                values = np.empty((len(time_positions), len(gids)), dtype=np.float32)
                print(f"{variable}: {len(time_positions)} hours x {len(gids)} counties")

                for column_start in range(0, len(gids), GID_CHUNK_SIZE):
                    column_end = min(column_start + GID_CHUNK_SIZE, len(gids))
                    gid_chunk = gids[column_start:column_end]
                    print(f"gids {column_start + 1}-{column_end} of {len(gids)}")

                    for row_index, time_position in enumerate(time_positions):
                        values[row_index, column_start:column_end] = resource[variable, int(time_position), gid_chunk]

                h5_file.create_dataset(variable, data=values, dtype="f4")

    county_h5_staging_rows.append({
        "weather_source": weather_source,
        "input_mode": "live HSDS",
        "hours": len(selected_times),
        "counties": len(gids),
        "path": output_h5.relative_to(Path("..")),
    })

display(pd.DataFrame(county_h5_staging_rows))

temperature_2m: 1 hours x 3143 counties
gids 1-800 of 3143


## Step 3: Calculate BA County Weather Weights

This step calculates the normalized county weights used to aggregate county weather into one BA-level weather series (MISO is used as an example here). The weight for each county is: `county_weight = county_population / total_population_of_BA_counties`

In [ ]:
BA_CODE = "MISO"

BA_MAPPING_CSV = PROVENANCE_DIR / "ba_service_territory_2019.csv"
COUNTY_POPULATION_CSV = PROVENANCE_DIR / "county_populations_2000_to_2020.csv"

ba_mapping = pd.read_csv(BA_MAPPING_CSV, dtype=str)
county_population = pd.read_csv(COUNTY_POPULATION_CSV, dtype={"county_FIPS": str})

# Use one consistent 5-digit county FIPS key in both input tables.
ba_mapping["county_fips"] = ba_mapping["County_FIPS"].str.split(".").str[0].str.zfill(5)
county_population["county_fips"] = county_population["county_FIPS"].str.zfill(5)

# Keep the counties assigned to the example BA.
ba_counties = ba_mapping.loc[ba_mapping["BA_Code"].eq(BA_CODE)].copy()

# Population weights tell Step 4 how much each county contributes to the BA average.
ba_county_weights = ba_counties.merge(county_population, on="county_fips")
ba_county_weights["normalized_ba_weather_weight"] = (ba_county_weights["pop_2020"] / ba_county_weights["pop_2020"].sum())

weight_check = pd.DataFrame([{
    "BA_Code": BA_CODE,
    "county_count": len(ba_county_weights),
    "sum_normalized_ba_weather_weight": ba_county_weights["normalized_ba_weather_weight"].sum(),
}])

display(ba_county_weights[["county_fips", "County_Name",  "State_Name", "pop_2020", "normalized_ba_weather_weight"]].head())
display(weight_check)


## Step 4: Combine County Weather Into Population-Weighted BA Weather

For each weather variable, the H5 file contains a table with one row per hour and one column per county. This step selects the county columns assigned to `BA_CODE` and weights their values into a single hourly weather series for the BA. Counties with larger populations contribute more to the BA average, using the normalized population weights calculated in Step 3. The calculation is repeated for every weather variable available from each source.

For a given hour, the population-weighted BA value is: `BA weather = sum(county weather value * county population weight)`. The county population weights sum to 1 within the BA.

In [ ]:
ba_weather_by_source = {}

for weather_source in WEATHER_SOURCES:
    h5_path = county_h5_by_source[weather_source]
    variables = SOURCE_VARIABLES[weather_source]

    with h5py.File(h5_path, "r") as h5_file:
        time_utc = pd.to_datetime(h5_file["time_utc"].asstr()[:], utc=True)

        # Match H5 county columns to the BA county weights from Step 3.
        h5_counties = pd.DataFrame({
            "county_fips": pd.Series(h5_file["county_fips"].asstr()[:]).str.zfill(5),
            "county_column": np.arange(len(h5_file["county_fips"])),
        })
        ba_county_columns = h5_counties.merge(ba_county_weights, on="county_fips")

        county_columns = ba_county_columns["county_column"].to_numpy(dtype=int)
        county_weights = ba_county_columns["normalized_ba_weather_weight"].to_numpy(dtype=float)

        # For each variable: BA_weather_t = sum(county_weather_i,t * county_weight_i).
        ba_source_weather = pd.DataFrame({"time_utc": time_utc})
        for variable in variables:
            county_weather = h5_file[variable][:, county_columns]
            ba_source_weather[variable] = (county_weather * county_weights).sum(axis=1)

    ba_weather_by_source[weather_source] = ba_source_weather

    print(f"{weather_source}: {len(county_columns)} {BA_CODE} counties, weight sum = {county_weights.sum():.6f}")
    display(ba_source_weather.head())

## Step 5: Merge BA Weather Tables And Write CSV

Step 5 combines the BA weather tables created in Step 4 into one final BA-weather table. This step is necessary when one dataset draws from multiple models at the same timestamp (ex: BC-HRRR and NSRDB), but it is not necessary when one dataset draws from one model only (ex: any of the sup3rCC models), as there is only one BA weather table created in Step 4. 

In this example, BC-HRRR supplies the meteorology variables and NSRDB supplies `ghi`. Both Step 4 outputs have the same hourly `time_utc` values, so this step merges them on `time_utc`, orders the native BA-weather columns, and writes the one-hour example BA-weather CSV.

The output CSV uses the same schema as the packaged 2007-2023 BA-weather archive and the load-forecast notebooks: `time_utc, temperature_2m, specifichumidity_2m, windspeed_10m, ghi, relativehumidity_2m, pressure_0m`

In [ ]:
ONE_HOUR_BA_WEATHER_CSV = OUTPUT_DIR / f"{BA_CODE}_{YEAR}_one_hour_ba_weather.csv"

# BC-HRRR supplies the meteorology variables; NSRDB supplies ghi.
bchrrr_ba_weather = ba_weather_by_source["bchrrr"]
nsrdb_ba_weather = ba_weather_by_source["nsrdb"]

# Merge the BA weather tables into the archived BA-weather column order.
ba_weather_example = bchrrr_ba_weather.merge(nsrdb_ba_weather, on="time_utc")
ba_weather_example = ba_weather_example[["time_utc", *NATIVE_WEATHER_COLUMNS]]

# Format time_utc like the packaged BA-weather CSVs.
ba_weather_example["time_utc"] = ba_weather_example["time_utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

ba_weather_example.to_csv(ONE_HOUR_BA_WEATHER_CSV, index=False)
print(f"Wrote one-hour example BA-weather CSV: {ONE_HOUR_BA_WEATHER_CSV.relative_to(Path('..'))}")
display(ba_weather_example)

## Downstream Use

The full operational BA-weather outputs created by this notebook are the `*_wtk_bchrrr_nsrdb_2007_2023_weather.csv` files consumed by the load-forecast notebooks.

# Appendices

Appendix A documents how WTK specific humidity is derived before aggregation. Appendix B repeats the same visible weighted-average calculation for Iowa counties.

## Appendix A: WTK Specific Humidity Derivation

WTK does not provide `specifichumidity_2m`, so WTK workflows derive it while writing the county-weather H5 file in Step 2. The derivation happens at the county-hour level before BA aggregation:

1. Read WTK `temperature_2m`, `relativehumidity_2m`, and `pressure_0m`.
2. Calculate `specifichumidity_2m` for each county and hour.
3. Write the derived `specifichumidity_2m` dataset into the WTK county-weather H5.
4. Use the same Step 4 population-weighting method as the other variables.

The specific humidity equations come from: https://www.cen.uni-hamburg.de/en/icdc/data/atmosphere/docs-atmo/ceop-derived-parameter-equations.pdf

In [ ]:
temperature_2m_c = 20
relativehumidity_2m_pct = 100
pressure_0m_pa = 101325

# Saturation vapor pressure over liquid water, in hPa.
e_s = 6.112 * np.exp((17.67 * temperature_2m_c) / (temperature_2m_c + 243.5))

# Actual vapor pressure from relative humidity.
e = e_s * relativehumidity_2m_pct / 100.0

# Specific humidity in kg kg-1.
specifichumidity_2m = (0.622 * e) / (pressure_0m_pa / 100.0 - 0.378 * e)
print(specifichumidity_2m)

## Appendix B: State County-Population Weather Aggregation

This process aggregates county-level weather data to the state-level (instead of BA-level), using Iowa as an example. It is analogous to step 4, but using all the counties in Iowa instead of all the counties in MISO.

In [ ]:
STATE_NAME = "Iowa"

# Select one unique row for every Iowa county.
iowa_county_weights = regrid_matches.loc[regrid_matches["state_name"].eq(STATE_NAME), ["county_fips"]].drop_duplicates().merge(county_population[["county_fips", "pop_2020"]], on="county_fips", validate="one_to_one")

# Normalize fixed 2020 population within Iowa.
iowa_county_weights["normalized_state_weather_weight"] = iowa_county_weights["pop_2020"] / iowa_county_weights["pop_2020"].sum()
state_weather_by_source = {}

for weather_source in WEATHER_SOURCES:
    with h5py.File(county_h5_by_source[weather_source], "r") as h5_file:
        h5_counties = pd.DataFrame({
            "county_fips": pd.Series(h5_file["county_fips"].asstr()[:]).str.zfill(5),
            "county_column": np.arange(len(h5_file["county_fips"])),
        })
        state_columns = h5_counties.merge(iowa_county_weights, on="county_fips", validate="one_to_one")
        columns = state_columns["county_column"].to_numpy(dtype=int)
        weights = state_columns["normalized_state_weather_weight"].to_numpy(dtype=float)
        state_source_weather = pd.DataFrame({"time_utc": pd.to_datetime(h5_file["time_utc"].asstr()[:], utc=True)})
        for variable in SOURCE_VARIABLES[weather_source]:
            state_source_weather[variable] = (h5_file[variable][:, columns] * weights).sum(axis=1)
    state_weather_by_source[weather_source] = state_source_weather

# Merge the source-specific results into the native seven-column schema.
iowa_weather_example = state_weather_by_source[WEATHER_SOURCES[0]]
for weather_source in WEATHER_SOURCES[1:]:
    iowa_weather_example = iowa_weather_example.merge(state_weather_by_source[weather_source], on="time_utc")
iowa_weather_example = iowa_weather_example[["time_utc", *NATIVE_WEATHER_COLUMNS]]
display(pd.DataFrame({"state": [STATE_NAME], "county_count": [len(iowa_county_weights)], "weight_sum": [iowa_county_weights["normalized_state_weather_weight"].sum()]}))
display(iowa_weather_example.head(1))